#Importing our Table configs 


In [0]:
base = "dbfs:/Volumes/dev_project/bronze/source_system"

# CRM folders
dbutils.fs.mkdirs(f"{base}/crm/cust_info")
dbutils.fs.mkdirs(f"{base}/crm/prd_info")
dbutils.fs.mkdirs(f"{base}/crm/sales_details")

# ERP folders
dbutils.fs.mkdirs(f"{base}/erp/CUST_AZ12")
dbutils.fs.mkdirs(f"{base}/erp/LOC_A101")
dbutils.fs.mkdirs(f"{base}/erp/PX_CAT_G1V2")

In [0]:
from bronze_config import INGESTION_CONFIG

#Write to bronze Layer


In [0]:
from pyspark.sql import functions as F

for item in INGESTION_CONFIG:

    table_name = f"dev_project.bronze.{item['table']}"

    # READ ALL FILES IN FOLDER 
    df = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .csv(item["path"])
        .withColumn("_ingestion_ts",  F.current_timestamp())
        .withColumn("_source",        F.lit(item["source"]))
        .withColumn("_file_name",     F.col("_metadata.file_path"))
    )

    # FILTER OUT FILES ALREADY INGESTED 
    if spark.catalog.tableExists(table_name):

        df_existing     = spark.table(table_name)
        processed_files = df_existing.select("_file_name").distinct()
        df              = df.join(processed_files, on="_file_name", how="left_anti")

    # WRITE ONLY NEW DATA 
    if not df.isEmpty():

        df.write \
            .format("delta") \
            .mode("append") \
            .saveAsTable(table_name)

        print(f"Loaded new data into {table_name}")

    else:
        print(f"No new files for {table_name}")